In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Lab_Hyperparameter_Tuning").master("local[*]").getOrCreate()

# Hyperparameter Tuning with Random Forests

In this lab, you will convert the Airbnb problem to a classification dataset, build a random forest classifier, and tune some hyperparameters of the random forest.

## Classification

In this lab, you will classify the listings between **high and low price.**

The **`class`** column will be:

- **`0`** for a low cost listing of under $150
- **`1`** for a high cost listing of $150 or more

The main difference with before is:
* We create a new column priceClass with the values from before
* We drop the price column

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

file_path = "/home/jovyan/work/datasets/airbnb/clean_data"

airbnb_df = (spark
            .read
            .parquet(file_path)
            .withColumn(<TODO>)
            .drop(<TODO>)
           )

train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)

categorical_cols = <TODO>
index_output_cols = <TODO>

string_indexer = StringIndexer(inputCols=<TODO>, outputCols=<TODO>, handleInvalid="skip")

numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType == "double") & (field != "priceClass"))]

assembler_inputs = <TODO>
vec_assembler = VectorAssembler(inputCols=<TODO>, outputCol="features")

## Random Forest

Create a Random Forest classifer called **`rf`** with the **`labelCol=priceClass`**, **`maxBins=250`**, and **`seed=42`** (for reproducibility).

It's under **`pyspark.ml.classification.RandomForestClassifier`** in Python.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(<TODO>)

## Grid Search

Let's define a grid of hyperparameters to test:
  - maxDepth: max depth of the decision tree (Use **`2, 5, 10`**)
  - numTrees: number of decision trees (Use **`10, 20, 100`**)

**`addGrid()`** accepts the name of the parameter (e.g. **`rf.maxDepth`**), and a list of the possible values (e.g. **`[2, 5, 10]`**).

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder

param_grid = (ParamGridBuilder()
              <TODO>
              <TODO>
              .build())

## Evaluator

In the past, we used a **`RegressionEvaluator`**. For classification, we can use a [BinaryClassificationEvaluator](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.BinaryClassificationEvaluator.html) if we have two classes or [MulticlassClassificationEvaluator](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.MulticlassClassificationEvaluator.html) for more than two classes.

Create a **`BinaryClassificationEvaluator`** with **`areaUnderROC`** as the metric.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol=<TODO>, metricName="areaUnderROC")

## Cross Validation

We are going to do **3-Fold** cross-validation and set the **`seed`**=42 on the cross-validator for reproducibility.

Put the Random Forest in the CV to speed up the [cross validation](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html) (as opposed to the pipeline in the CV).

In [ ]:
from pyspark.ml.tuning import CrossValidator

cv = CrossValidator(estimator=<TODO>,
                    evaluator=<TODO>,
                    estimatorParamMaps=<TODO>,
                    numFolds=<TODO>, seed=42)

## Pipeline

Let's fit the pipeline with our cross validator to our training data (this may take a few minutes).

In [ ]:
stages = [<TODO>]

pipeline = Pipeline(stages=<TODO>)

pipeline_model = pipeline.fit(<TODO>)

## Hyperparameter

Which hyperparameter combination performed the best?

In [ ]:
cv_model = pipeline_model.stages[-1]
rf_model = cv_model.bestModel

# list(zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics))

print(rf_model.explainParams())

## Feature Importance

In [ ]:
import pandas as pd

pandas_df = pd.DataFrame(list(zip(vec_assembler.getInputCols(), rf_model.featureImportances)), columns=["feature", "importance"])
top_features = pandas_df.sort_values(["importance"], ascending=False)
top_features

Do those features make sense? Would you use those features when picking an Airbnb rental?

## Apply Model to Test Set

In [ ]:
pred_df = pipeline_model.transform(<TODO>)
area_under_roc = evaluator.evaluate(<TODO>)
print(f"Area under ROC is {area_under_roc:.2f}")

#You should get 0.88